# LoVoCCS morphology: Checking for like galaxy clusters in the whole morphology parameter space

<span style="color:red">This section of the project...</span>

## Main takeaways 

In summary:



## Import Statements

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
from matplotlib import pyplot as plt
import os
from astropy.units import Quantity

# This adds the directory above to the path, allowing me to import the common functions that I've written in
#  common.py - this just saves me repeating boring code and makes sure its all consistent
import sys
sys.path.insert(0, '../')
from common import lovoccs_cosmo, haversine

# We could just re-implement this, but it is already there in XGA
from xga.sourcetools.misc import ang_to_rad

%matplotlib inline

## Useful values

Here we set up any values that are useful to several parts of the analysis in this notebook, and that we might wish to change in the future:

In [2]:
# 

## Output paths

Paths to save comparison figures and any output results files - defined here to avoid repetition and make the code easier to read: 

In [3]:
# sep_fig_path = "../../outputs/figures/positions_and_morphology/coordinate_offsets/"
# os.makedirs(sep_fig_path, exist_ok=True)

# sep_res_path = "../../outputs/result_files/positions_and_morphology/"
# os.makedirs(sep_res_path, exist_ok=True)

## Cosmological model

We employ the same cosmological model utilized in the LoVoCCS-I & II analyses (Fu et al. [2022](https://ui.adsabs.harvard.edu/abs/2022ApJ...933...84F/abstract), [2024](https://ui.adsabs.harvard.edu/abs/2024ApJ...974...69F/abstract)):

In [4]:
lovoccs_cosmo

LambdaCDM(name=None, H0=<Quantity 71. km / (Mpc s)>, Om0=0.2648, Ode0=0.7352, Tcmb0=<Quantity 0. K>, Neff=3.04, m_nu=None, Ob0=0.0448)

## Loading data files

Important considerations for this dataset:

* Some clusters selected for LoVoCCS have been identified as multiple blended systems in the course of the X-LoVoCCS project - other LoVoCCS works may only have entries/have made measurements for the overall system.
* Not all LoVoCCS-II galaxy clusters have XMM data available - as such we will not yet have measured X-ray centroids for them.

### X-LoVoCCS-I base sample

In [5]:
xlovoccs_base = pd.read_csv("../../sample_files/X-LoVoCCSI.csv")
xlovoccs_base.insert(0, 'name', xlovoccs_base['LoVoCCSID'].apply(lambda x: "LoVoCCS-" + str(x)))
xlovoccs_base.head(6)

,name,LoVoCCSID,parent_LoVoCCSID,Name,start_ra,start_dec,MCXC_Redshift,MCXC_R500,MCXC_RA,MCXC_DEC,manual_xray_ra,manual_xray_dec,MCXC_Lx500_0.1_2.4
0,LoVoCCS-1,1,1,A2029,227.734300,5.745471,0.0766,1.3344,227.73000,5.720000,227.734300,5.745471,8.726709e+44
1,LoVoCCS-2,2,2,A401,44.740000,13.580000,0.0739,1.2421,44.74000,13.580000,NaN,NaN,6.088643e+44
2,LoVoCCS-4A,4A,4,A85North,10.458750,-9.301944,0.0555,1.2103,10.45875,-9.301944,NaN,NaN,5.100085e+44
3,LoVoCCS-4B,4B,4,A85South,10.451487,-9.460007,0.0555,1.2103,10.45875,-9.301944,10.451487,-9.460007,5.100085e+44
4,LoVoCCS-5,5,5,A3667,303.157313,-56.845978,0.0556,1.1990,303.13000,-56.830000,303.157313,-56.845978,4.871933e+44
5,LoVoCCS-7,7,7,A3827,330.480000,-59.950000,0.0980,1.1367,330.48000,-59.950000,NaN,NaN,4.204419e+44


### X-LoVoCCS-I component alignments

In [ ]:
xlovoccs_align = pd.read_csv("")

### X-LoVoCCS-I component separations

### X-LoVoCCS-I morphologies

### X-LoVoCCS-I axis ratios

### Combining tables

In [9]:
xlovoccs_samp = pd.merge(xlovoccs_base, xlovoccs_centroid, left_on='name', right_on='name', how='outer')
xlovoccs_samp = pd.merge(xlovoccs_samp, xlovoccs_peak, left_on='name', right_on='name', how='outer')
xlovoccs_samp = pd.merge(xlovoccs_samp, lovoccsII_morph, left_on='parent_LoVoCCSID', right_on='parent_LoVoCCSID', how='inner')
xlovoccs_samp = xlovoccs_samp.sort_values('parent_LoVoCCSID').reset_index(drop=True)
xlovoccs_samp.head(6)

,name,LoVoCCSID,parent_LoVoCCSID,Name,start_ra,start_dec,MCXC_Redshift,MCXC_R500,MCXC_RA,MCXC_DEC,manual_xray_ra,manual_xray_dec,MCXC_Lx500_0.1_2.4,cent_ra,cent_ra-,cent_ra+,cent_dec,cent_dec-,cent_dec+,position_angle,position_angle-,position_angle+,ax_ratio,ax_ratio-,ax_ratio+,peak_ra,peak_dec,cluster,redshift_target,BCG_ra,BCG_dec,mass_map_ra,mass_map_dec,RS_map_ra,RS_map_dec,BCG_angle,mass_map_angle_05,mass_map_angle_10,mass_map_angle_20,RS_map_angle_05,RS_map_angle_10,RS_map_angle_20,cluster_type
0,LoVoCCS-1,1,1.0,A2029,227.734300,5.745471,0.0766,1.3344,227.73000,5.720000,227.734300,5.745471,8.726709e+44,227.73324,0.00002,0.00002,5.74338,0.00003,0.00003,-73.0512,0.2030,0.2157,0.9202,0.0011,0.0012,227.733253,5.745184,A2029,0.0773,227.733771,5.744819,227.731704,5.775644,227.742692,5.770095,-62.61,84.602196,83.570516,-69.689253,-74.378012,-74.170775,-84.280216,a
1,LoVoCCS-2,2,2.0,A401,44.740000,13.580000,0.0739,1.2421,44.74000,13.580000,NaN,NaN,6.088643e+44,44.73661,0.00017,0.00015,13.57730,0.00018,0.00016,23.7327,4.3632,3.8393,0.9746,0.0049,0.0043,44.750675,13.594567,A401,0.0743,44.740801,13.582686,44.741462,13.582399,44.742694,13.579045,-63.16,-6.824412,0.640114,-24.654011,-61.347490,-47.585406,-58.138832,a
2,LoVoCCS-4B,4B,4.0,A85South,10.451487,-9.460007,0.0555,1.2103,10.45875,-9.301944,10.451487,-9.460007,5.100085e+44,10.46330,0.00018,0.00016,-9.43280,0.00024,0.00029,-71.9520,0.9161,1.0538,0.9176,0.0047,0.0038,NaN,NaN,A85,0.0556,10.460319,-9.303484,10.446739,-9.332133,10.405090,-9.312744,58.59,-70.095509,-84.031952,-40.610942,84.708468,72.220682,74.508138,b
3,LoVoCCS-4A,4A,4.0,A85North,10.458750,-9.301944,0.0555,1.2103,10.45875,-9.301944,NaN,NaN,5.100085e+44,10.46006,0.00003,0.00004,-9.30457,0.00004,0.00004,67.2727,0.2179,0.1952,0.8910,0.0009,0.0010,NaN,NaN,A85,0.0556,10.460319,-9.303484,10.446739,-9.332133,10.405090,-9.312744,58.59,-70.095509,-84.031952,-40.610942,84.708468,72.220682,74.508138,b
4,LoVoCCS-5,5,5.0,A3667,303.157313,-56.845978,0.0556,1.1990,303.13000,-56.830000,303.157313,-56.845978,4.871933e+44,303.14189,0.00014,0.00015,-56.83829,0.00009,0.00007,40.0924,0.2190,0.2355,0.8262,0.0011,0.0012,303.100675,-56.851435,A3667,0.0552,303.113931,-56.826788,303.135914,-56.845641,303.150738,-56.836339,44.27,-68.929259,79.595996,-84.325048,38.482847,44.175107,48.139765,a
5,LoVoCCS-7,7,7.0,A3827,330.480000,-59.950000,0.0980,1.1367,330.48000,-59.950000,NaN,NaN,4.204419e+44,330.47766,0.00011,0.00024,-59.94796,0.00011,0.00011,-78.3924,0.9435,0.8295,0.9519,0.0032,0.0037,330.466217,-59.948751,A3827,0.0972,330.469621,-59.946597,330.458667,-59.942374,330.481129,-59.937518,72.40,78.181991,68.843089,56.923349,72.853008,64.101562,83.362138,a


## <span style="color:red">BFCornerPlot</span>

<span style="color:red">...</span>